# Week 8 · Topic 4 — From Functions to a Class (optional extra)

**Box we are building: the project's real architecture, starting now**

The same refactor Week 5's demo did to `record_check.py`. This time you do it
to your own code.

> **Already know this?** Skip to the *Topic 4 Challenge*.

> **Optional extra.** Your CW2 project uses plain functions, not classes, so you do not need this topic for the lab. It shows how the same code looks when it is bundled into a class.

---
## 1. Before: loose functions and a stray variable

In [3]:
def parse_line(raw_line):
    """Turn one messy raw line into a dict."""
    label, value = raw_line.strip().split(",")
    return {"label": label.strip().lower(), "value": float(value)}

def status_of(value, limit=100):
    """Return OVER LIMIT or OK for a value against a limit."""
    return "OVER LIMIT" if value >= limit else "OK"

over_limit_count = 0   # <- not attached to anything

### Your turn
Call `parse_line` and `status_of` on `"srv-01, 87"` and print the result.

In [4]:
parsed = parse_line("srv-01, 87")
print(parsed)
print(status_of(parsed["value"]))

{'label': 'srv-01', 'value': 87.0}
OK


---
## 2. After: one class, bundling the same data and logic

Nothing here is new — every line is something Week 5 already taught. What
changed is where it lives.

In [5]:
class RecordSource:
    """Loads raw lines, parses them, and tracks how many are over limit."""

    def __init__(self, limit=100):
        self.limit = limit
        self.records = []
        self.over_limit_count = 0

    def parse_line(self, raw_line):
        label, value = raw_line.strip().split(",")
        return {"label": label.strip().lower(), "value": float(value)}

    def status_of(self, value):
        return "OVER LIMIT" if value >= self.limit else "OK"

```
before: loose functions            after: one class

parse_line(raw_line)                class RecordSource:
status_of(value, limit)                 self.limit
over_limit_count  (stray)               self.over_limit_count
                                         def parse_line(self, raw_line): ...
                                         def status_of(self, value): ...

same logic - now it travels together, and two RecordSource objects
never interfere with each other
```

### Your turn
Add a `load(self, raw_lines)` method that parses every line, prints its status, and updates `self.over_limit_count`.

In [6]:
class RecordSource:
    
    def __init__(self, limit=100):
        self.limit = limit
        self.records = []
        self.over_limit_count = 0

    def parse_line(self, raw_line):
        label, value = raw_line.strip().split(",")
        return {"label": label.strip().lower(), "value": float(value)}

    def status_of(self, value):
        return "OVER LIMIT" if value >= self.limit else "OK"

    def load(self, raw_lines):
        for raw_line in raw_lines:
            record = self.parse_line(raw_line)
            self.records.append(record)
            status = self.status_of(record["value"])
            print(f"{record['label']} {status}")
            if status == "OVER LIMIT":
                self.over_limit_count += 1

---
## 3. What actually changed

`over_limit_count` can no longer be forgotten or reset by accident — it
belongs to the object. `parse_line` and `status_of` no longer need their
inputs passed between calls — they share `self`. You can now make a *second*
`RecordSource` with a different limit, and the two will not interfere.

In [7]:
a = RecordSource(limit=100)
b = RecordSource(limit=50)
print(a.limit, b.limit)   # independent

100 50


### Your turn
Create two `RecordSource` objects with different limits and prove they do not share state.

In [10]:
a = RecordSource(limit=100)
b = RecordSource(limit=50)
a.load(["srv-01, 87"])
b.load(["srv-01, 87"])
print(a.over_limit_count, b.over_limit_count)

srv-01 OK
srv-01 OVER LIMIT
0 1


---
## Topic 4 Challenge

Take your Week 7 `parse_line`/`status_of` functions (or the versions above)
and refactor them into one class, exactly the way this topic did — this is
literally this week's project task, rehearsed here first.

In [11]:
class RecordChecker:
    def __init__(self, limit=100, warning_limit=90):
        self.limit = limit
        self.warning_limit = warning_limit

    def parse_line(self, raw_line):
        raw_line = raw_line.strip()
        hostname_part, rest = raw_line.split(",", 1)
        hostname = hostname_part.strip().lower()
        fields = [part.replace(" ", "") for part in rest.split(",")]
        used = fields[0].split("=")
        total = fields[1].split("=")
        
        return {"hostname": hostname, used[0]: float(used[1]), total[0]: float(total[1])}

    def status_of(self, record):
        percent = record["used"] / record["total"] * 100
        if percent >= self.limit:
            return "OVER LIMIT"
        elif percent >= self.warning_limit:
            return "WARNING"
        return "OK"

checker = RecordChecker()
record = checker.parse_line("  Srv-01 , used=93 , total=100 ")
print(record)
print(checker.status_of(record))

{'hostname': 'srv-01', 'used': 93.0, 'total': 100.0}
WARNING


---
## Same idea, three fields

| Field | Looks like |
|---|---|
| **AI / Data Science** | a `Dataset` class bundling loading, cleaning and summarising |
| **Cyber Security** | an `IncidentLog` class bundling loading, parsing and flagging |
| **IT** | an `OpsConsole` class bundling fetching, checking and reporting |

---
## Common mistakes

| Mistake | What you see | Fix |
|---|---|---|
| forgetting to change `value` to `self.value` when moving a function in | `NameError` or wrong result | every reference to shared data needs `self.` |
| leaving `limit` as a function parameter instead of `self.limit` | works, but defeats the point — nothing is bundled | store it in `__init__`, read it via `self` |
| one class doing everything with no clear responsibility | hard to test, hard to extend | keep it to the same "one job" rule as functions had |
| reaching for inheritance or a second class | more complexity than this project needs yet | one or two classes is the whole ask this week |

## Go deeper

`examples/01_before_and_after.py` — the exact refactor, side by side
`examples/02_a_class_that_pulls_real_data.py` — the same idea as functions that share settings (optional topic: the lab itself uses plain functions)

---
## Recap

| You can now | Using |
|---|---|
| spot when functions should become a class | shared data, passed around too much |
| do the refactor | bundle data in `__init__`, methods use `self` |
| keep two instances independent | separate objects, separate state |

**You now have all of Week 8's tools.** This is what starts your project in
the lab. The project you start there is the one you keep building until Week 12.